# Skin Cancer Classification from Dermatoscopic Images
**APS360 — Final Report**

This notebook covers:
1. Environment setup & imports
2. Dataset download (HAM10000 via Kaggle)
3. Data exploration & class distribution
4. Sample image visualization
5. Data split (stratified, leak-free)
6. Baseline model (Random Forest on HOG + colour features)
7. Primary model (EfficientNet-B3, two-phase fine-tuning + label smoothing)
8. Validation evaluation
9. **Test set evaluation (held-out)**
10. **Test-time augmentation (TTA)**
11. **Grad-CAM visualizations**
12. **ISIC 2019 out-of-distribution evaluation**
13. **Inference function**

## 1. Install Dependencies

In [ ]:
!pip install kaggle scikit-image --quiet

## 2. Kaggle API Setup


In [ ]:
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API key configured.')

## 3. Download HAM10000 Dataset

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --quiet
!unzip -q skin-cancer-mnist-ham10000.zip -d ham10000
print('Dataset downloaded and extracted.')

## 4. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from skimage.feature import hog
from skimage.color import rgb2hsv

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 5. Load Metadata

In [ ]:
df = pd.read_csv('ham10000/HAM10000_metadata.csv')
print(f'Total images: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

## 6. Class Distribution

In [ ]:
LABEL_MAP = {
    'nv':    'Melanocytic Nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratoses',
    'vasc':  'Vascular Lesions',
    'df':    'Dermatofibroma',
}
CLASS_ORDER = list(LABEL_MAP.keys())
COLORS = ['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f','#edc948','#b07aa1']

counts = df['dx'].value_counts().reindex(CLASS_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
bars = axes[0].bar(CLASS_ORDER, counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
axes[0].set_title('Class Distribution — HAM10000', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lesion Type (abbreviation)')
axes[0].set_ylabel('Number of Images')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 40, str(v),
                 ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=CLASS_ORDER, colors=COLORS,
            autopct='%1.1f%%', startangle=140, pctdistance=0.78)
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass counts:')
for k, v in zip(counts.index, counts.values):
    print(f'  {k.upper():6s} {LABEL_MAP[k]:<25s}: {v:5d} ({v/len(df)*100:.1f}%)')

## 7. Sample Image Visualization

In [ ]:
IMG_DIRS = [
    'ham10000/ham10000_images_part_1',
    'ham10000/ham10000_images_part_2',
]

def find_image(image_id):
    for d in IMG_DIRS:
        p = os.path.join(d, image_id + '.jpg')
        if os.path.exists(p):
            return p
    return None

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
fig.suptitle('One Sample per Lesion Class', fontsize=13, fontweight='bold', y=1.02)

for ax, (dx, name), color in zip(axes, LABEL_MAP.items(), COLORS):
    row = df[df['dx'] == dx].sample(1, random_state=SEED).iloc[0]
    path = find_image(row['image_id'])
    if path:
        ax.imshow(Image.open(path))
    ax.set_title(dx.upper(), fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel(name, fontsize=7, wrap=True)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Data Split (Stratified, Leak-Free)
Images sharing a `lesion_id` (same patient) are confined to one split.

In [ ]:
# Group at lesion level
lesion_df = df.groupby('lesion_id')['dx'].first().reset_index()

train_ids, temp_ids = train_test_split(
    lesion_df['lesion_id'], test_size=0.30,
    stratify=lesion_df['dx'], random_state=SEED)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50,
    stratify=lesion_df[lesion_df['lesion_id'].isin(temp_ids)]['dx'],
    random_state=SEED)

train_df = df[df['lesion_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['lesion_id'].isin(val_ids)].reset_index(drop=True)
test_df  = df[df['lesion_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train : {len(train_df):5d} images')
print(f'Val   : {len(val_df):5d} images')
print(f'Test  : {len(test_df):5d} images')

# Verify no lesion_id overlap
assert len(set(train_df.lesion_id) & set(val_df.lesion_id)) == 0
assert len(set(train_df.lesion_id) & set(test_df.lesion_id)) == 0
print('No patient-level leakage confirmed.')

---
## 9. Baseline Model — Random Forest on HOG + Colour Features

In [ ]:
IMG_SIZE = 64   # smaller size for fast HOG extraction

def extract_features(image_id):
    """Returns 1788-dim feature vector: colour stats (12) + HOG (1764)."""
    path = find_image(image_id)
    img = np.array(Image.open(path).resize((IMG_SIZE, IMG_SIZE))) / 255.0

    # Colour features: mean + std per RGB and HSV channel
    hsv = rgb2hsv(img)
    colour_feats = np.concatenate([
        img.mean(axis=(0,1)), img.std(axis=(0,1)),
        hsv.mean(axis=(0,1)), hsv.std(axis=(0,1))
    ])  # 12 values

    # HOG features
    hog_feats = hog(img, orientations=9, pixels_per_cell=(8,8),
                    cells_per_block=(2,2), channel_axis=-1)

    return np.concatenate([colour_feats, hog_feats])

print('Extracting features for train set (this takes ~2 min)...')
X_train = np.array([extract_features(iid) for iid in train_df['image_id']])
y_train = train_df['dx'].map(CLASS_ORDER.index).values

print('Extracting features for val set...')
X_val = np.array([extract_features(iid) for iid in val_df['image_id']])
y_val = val_df['dx'].map(CLASS_ORDER.index).values

print(f'Feature vector size: {X_train.shape[1]}')

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)
print('Random Forest trained.')

y_pred_rf = rf.predict(X_val)

rf_acc      = (y_pred_rf == y_val).mean()
rf_bal_acc  = balanced_accuracy_score(y_val, y_pred_rf)
rf_macro_f1 = f1_score(y_val, y_pred_rf, average='macro')

print(f'\nRandom Forest — Validation Results')
print(f'  Overall accuracy  : {rf_acc:.4f}')
print(f'  Balanced accuracy : {rf_bal_acc:.4f}')
print(f'  Macro F1          : {rf_macro_f1:.4f}')
print()
print(classification_report(y_val, y_pred_rf, target_names=[c.upper() for c in CLASS_ORDER]))

In [ ]:
# Per-class F1 bar chart + confusion matrix
f1_per_class = f1_score(y_val, y_pred_rf, average=None)
cm_rf = confusion_matrix(y_val, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar([c.upper() for c in CLASS_ORDER], f1_per_class, color=COLORS, edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_title('Baseline RF — Per-Class F1 (Val)', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('F1 Score')
axes[0].axhline(rf_macro_f1, color='black', linestyle='--', linewidth=1, label=f'Macro F1 = {rf_macro_f1:.2f}')
axes[0].legend(fontsize=9)
for i, v in enumerate(f1_per_class):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)

disp = ConfusionMatrixDisplay(cm_rf, display_labels=[c.upper() for c in CLASS_ORDER])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Baseline RF — Confusion Matrix (Val)', fontweight='bold')

plt.tight_layout()
plt.savefig('baseline_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Primary Model — EfficientNet-B3 (Transfer Learning)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE_CNN  = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE_CNN, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = find_image(row['image_id'])
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = CLASS_ORDER.index(row['dx'])
        return img, label

# ── Weighted sampler to handle class imbalance ────────────────────────────────
train_labels = train_df['dx'].map(CLASS_ORDER.index).values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds   = SkinLesionDataset(val_df,   transform=val_transform)
test_ds  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
def build_efficientnet_b3(num_classes=7, dropout=0.3):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    in_features = model.classifier[1].in_features  # 1536
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_efficientnet_b3().to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_head  = sum(p.numel() for p in model.classifier.parameters())
print(f'Total parameters       : {total_params:,}')
print(f'Trainable (head only)  : {trainable_head:,}')

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
ce_weights = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float32).to(DEVICE)
criterion  = nn.CrossEntropyLoss(weight=ce_weights)

def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('classifier')

def unfreeze_top_blocks(model, n_blocks=2):
    """Unfreeze the last n blocks of EfficientNet features."""
    for p in model.parameters():
        p.requires_grad = False
    # Classifier always trainable
    for p in model.classifier.parameters():
        p.requires_grad = True
    # Unfreeze last n_blocks of model.features
    blocks = list(model.features.children())
    for block in blocks[-n_blocks:]:
        for p in block.parameters():
            p.requires_grad = True

def run_epoch(loader, model, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            n += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    return total_loss / n, correct / n, bal_acc

In [ ]:
# ── Phase 1: Feature Extraction (backbone frozen, head only) ──────────────────
PHASE1_EPOCHS = 10

freeze_backbone(model)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[],
           'train_bal':[], 'val_bal':[]}
best_val_bal = 0.0

print('=== Phase 1: Feature Extraction (backbone frozen) ===')
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc, tr_bal = run_epoch(train_loader, model, optimizer, train=True)
    vl_loss, vl_acc, vl_bal = run_epoch(val_loader,   model, train=False)
    scheduler.step()

    for k, v in zip(['train_loss','val_loss','train_acc','val_acc','train_bal','val_bal'],
                    [tr_loss, vl_loss, tr_acc, vl_acc, tr_bal, vl_bal]):
        history[k].append(v)

    if vl_bal > best_val_bal:
        best_val_bal = vl_bal
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'  Epoch {epoch:2d}/{PHASE1_EPOCHS} | '
          f'Train loss {tr_loss:.4f} acc {tr_acc:.3f} bal {tr_bal:.3f} | '
          f'Val loss {vl_loss:.4f} acc {vl_acc:.3f} bal {vl_bal:.3f}')

In [ ]:
# ── Phase 2: Fine-Tuning (top 2 backbone blocks + head) ──────────────────────
PHASE2_EPOCHS = 15

unfreeze_top_blocks(model, n_blocks=2)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

print('=== Phase 2: Fine-Tuning (top 2 backbone blocks unfrozen) ===')
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc, tr_bal = run_epoch(train_loader, model, optimizer, train=True)
    vl_loss, vl_acc, vl_bal = run_epoch(val_loader,   model, train=False)
    scheduler.step()

    for k, v in zip(['train_loss','val_loss','train_acc','val_acc','train_bal','val_bal'],
                    [tr_loss, vl_loss, tr_acc, vl_acc, tr_bal, vl_bal]):
        history[k].append(v)

    if vl_bal > best_val_bal:
        best_val_bal = vl_bal
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'  Epoch {epoch:2d}/{PHASE2_EPOCHS} | '
          f'Train loss {tr_loss:.4f} acc {tr_acc:.3f} bal {tr_bal:.3f} | '
          f'Val loss {vl_loss:.4f} acc {vl_acc:.3f} bal {vl_bal:.3f}')

print(f'\nBest val balanced accuracy: {best_val_bal:.4f}')

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
epochs = list(range(1, total_epochs + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train', color='#4e79a7', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val',   color='#f28e2b', linewidth=2, linestyle='--')
axes[0].axvline(PHASE1_EPOCHS + 0.5, color='red', linestyle=':', linewidth=1.5, label='Fine-tune start')
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Balanced accuracy
axes[1].plot(epochs, history['train_bal'], label='Train', color='#4e79a7', linewidth=2)
axes[1].plot(epochs, history['val_bal'],   label='Val',   color='#f28e2b', linewidth=2, linestyle='--')
axes[1].axvline(PHASE1_EPOCHS + 0.5, color='red', linestyle=':', linewidth=1.5, label='Fine-tune start')
axes[1].set_title('Balanced Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Balanced Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Evaluation — Best Model on Validation Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cnn_acc     = (np.array(all_preds) == np.array(all_labels)).mean()
cnn_bal_acc = balanced_accuracy_score(all_labels, all_preds)
cnn_macro_f1 = f1_score(all_labels, all_preds, average='macro')

print('=== EfficientNet-B3 — Validation Results ===')
print(f'  Overall accuracy  : {cnn_acc:.4f}')
print(f'  Balanced accuracy : {cnn_bal_acc:.4f}')
print(f'  Macro F1          : {cnn_macro_f1:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=[c.upper() for c in CLASS_ORDER]))

In [ ]:
# Confusion matrix + per-class F1 comparison
cnn_f1_per_class = f1_score(all_labels, all_preds, average=None)
rf_f1_per_class  = f1_score(y_val, y_pred_rf, average=None)
cm_cnn = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-class F1 comparison
x = np.arange(len(CLASS_ORDER))
w = 0.35
axes[0].bar(x - w/2, rf_f1_per_class,  w, label='RF Baseline', color='#aec7e8', edgecolor='white')
axes[0].bar(x + w/2, cnn_f1_per_class, w, label='EfficientNet-B3', color='#4e79a7', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.upper() for c in CLASS_ORDER])
axes[0].set_ylim(0, 1)
axes[0].set_title('Per-Class F1: Baseline vs EfficientNet-B3', fontweight='bold')
axes[0].set_ylabel('F1 Score')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# CNN confusion matrix
disp = ConfusionMatrixDisplay(cm_cnn, display_labels=[c.upper() for c in CLASS_ORDER])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('EfficientNet-B3 — Confusion Matrix (Val)', fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary comparison table
print('\n=== Summary Comparison ===')
print(f'{"Metric":<25} {"RF Baseline":>12} {"EfficientNet-B3":>16} {"Delta":>8}')
print('-' * 65)
for metric, rf_v, cnn_v in [
    ('Overall Accuracy',   rf_acc,      cnn_acc),
    ('Balanced Accuracy',  rf_bal_acc,  cnn_bal_acc),
    ('Macro F1',           rf_macro_f1, cnn_macro_f1),
]:
    print(f'{metric:<25} {rf_v:>12.4f} {cnn_v:>16.4f} {cnn_v - rf_v:>+8.4f}')

---
## 12. Test Set Evaluation (Held-Out)
The test set is **never touched during training or hyperparameter tuning**.

In [ ]:
# Load best checkpoint and evaluate on held-out test set
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(labels.numpy())

test_acc      = (np.array(test_preds) == np.array(test_labels)).mean()
test_bal_acc  = balanced_accuracy_score(test_labels, test_preds)
test_macro_f1 = f1_score(test_labels, test_preds, average='macro')

print('=== EfficientNet-B3 — Held-Out Test Set Results ===')
print(f'  Overall accuracy  : {test_acc:.4f}')
print(f'  Balanced accuracy : {test_bal_acc:.4f}')
print(f'  Macro F1          : {test_macro_f1:.4f}')
print()
print(classification_report(test_labels, test_preds,
                            target_names=[c.upper() for c in CLASS_ORDER]))

# Confusion matrix
cm_test = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm_test, display_labels=[c.upper() for c in CLASS_ORDER])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('EfficientNet-B3 — Confusion Matrix (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig('test_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. Test-Time Augmentation (TTA)
Average softmax predictions over multiple augmented views of each test image to improve robustness.

In [ ]:
tta_transforms = [
    val_transform,  # original
    transforms.Compose([
        transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
        transforms.RandomRotation((90, 90)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
        transforms.RandomRotation((180, 180)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]

def predict_with_tta(model, dataset_df, tta_transforms, device):
    model.eval()
    all_probs = []
    for tfm in tta_transforms:
        ds  = SkinLesionDataset(dataset_df, transform=tfm)
        loader = DataLoader(ds, batch_size=32, shuffle=False,
                            num_workers=2, pin_memory=True)
        probs = []
        with torch.no_grad():
            for imgs, _ in loader:
                imgs = imgs.to(device)
                out  = torch.softmax(model(imgs), dim=1).cpu().numpy()
                probs.extend(out)
        all_probs.append(np.array(probs))
    # Average probabilities across all TTA views
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs.argmax(axis=1)

tta_preds  = predict_with_tta(model, test_df, tta_transforms, DEVICE)
tta_labels = test_df['dx'].map(CLASS_ORDER.index).values

tta_acc     = (tta_preds == tta_labels).mean()
tta_bal_acc = balanced_accuracy_score(tta_labels, tta_preds)
tta_f1      = f1_score(tta_labels, tta_preds, average='macro')

print('=== TTA Results (Test Set, 5 views) ===')
print(f'  Overall accuracy  : {tta_acc:.4f}  (baseline: {test_acc:.4f})')
print(f'  Balanced accuracy : {tta_bal_acc:.4f}  (baseline: {test_bal_acc:.4f})')
print(f'  Macro F1          : {tta_f1:.4f}  (baseline: {test_macro_f1:.4f})')

---
## 14. Grad-CAM Visualizations
Grad-CAM highlights which image regions most influence each prediction, helping verify the model attends to the lesion rather than background artefacts.

In [ ]:
import torch.nn.functional as F

class GradCAM:
    """Grad-CAM for EfficientNet-B3 — hooks onto the last conv block."""
    def __init__(self, model):
        self.model    = model
        self.gradients = None
        self.activations = None
        # Hook onto the last feature block of EfficientNet
        target_layer = model.features[-1]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        output[0, class_idx].backward()
        # Pool gradients across spatial dimensions
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


def show_gradcam(model, image_ids, labels, n=6):
    gradcam = GradCAM(model)
    fig, axes = plt.subplots(2, n, figsize=(n * 2.8, 5.5))
    fig.suptitle('Grad-CAM — Top row: original | Bottom row: activation heatmap',
                 fontsize=11, fontweight='bold')

    # Sample: mix of correct and incorrect predictions
    sample_ids   = image_ids[:n]
    sample_labels = labels[:n]

    for col, (img_id, true_label) in enumerate(zip(sample_ids, sample_labels)):
        path = find_image(img_id)
        pil_img = Image.open(path).convert('RGB')
        inp = val_transform(pil_img).unsqueeze(0).to(DEVICE)
        inp.requires_grad_(True)

        cam, pred_idx = gradcam.generate(inp)
        pred_label = CLASS_ORDER[pred_idx]
        correct = (pred_idx == true_label)
        border  = 'green' if correct else 'red'

        # Original image
        axes[0, col].imshow(pil_img)
        axes[0, col].set_title(
            f'True: {CLASS_ORDER[true_label].upper()}\nPred: {pred_label.upper()}',
            fontsize=8,
            color=border
        )
        axes[0, col].axis('off')
        for spine in axes[0, col].spines.values():
            spine.set_edgecolor(border); spine.set_linewidth(2)

        # Grad-CAM overlay
        axes[1, col].imshow(pil_img)
        axes[1, col].imshow(cam, cmap='jet', alpha=0.45)
        axes[1, col].axis('off')

    plt.tight_layout()
    plt.savefig('gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()


# Show Grad-CAM for a sample of test images
sample_test = test_df.sample(6, random_state=42)
show_gradcam(
    model,
    sample_test['image_id'].tolist(),
    sample_test['dx'].map(CLASS_ORDER.index).tolist()
)

---
## 15. ISIC 2019 — Out-of-Distribution Evaluation
Evaluate on new patients from different clinical sites to test generalisation beyond HAM10000.

In [ ]:
# Download ISIC 2019 test images
# Note: ISIC 2019 uses the same 7 overlapping classes as HAM10000
!kaggle datasets download -d andrewmvd/isic-2019 --quiet 2>/dev/null || \
 echo 'ISIC 2019 not on Kaggle — download from https://challenge.isic-archive.com/data/#2019'

# If download succeeded, extract and evaluate
import os
if os.path.exists('isic-2019.zip'):
    !unzip -q isic-2019.zip -d isic2019
    isic_meta = pd.read_csv('isic2019/ISIC_2019_Training_GroundTruth.csv')

    # Map ISIC 2019 column names to HAM10000 dx codes
    isic_col_map = {
        'NV': 'nv', 'MEL': 'mel', 'BKL': 'bkl',
        'BCC': 'bcc', 'AK': 'akiec', 'VASC': 'vasc', 'DF': 'df'
    }
    label_cols = [c for c in isic_col_map if c in isic_meta.columns]
    isic_meta['dx'] = isic_meta[label_cols].idxmax(axis=1).map(isic_col_map)
    isic_meta = isic_meta[isic_meta['dx'].isin(CLASS_ORDER)]

    # Sample up to 500 images per available class for speed
    isic_sample = isic_meta.groupby('dx', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 500), random_state=42)
    ).reset_index(drop=True)

    # Build a minimal dataset pointing to ISIC 2019 images
    ISIC_IMG_DIR = 'isic2019/ISIC_2019_Training_Input'

    class ISIC2019Dataset(Dataset):
        def __init__(self, df, img_dir, transform):
            self.df = df.reset_index(drop=True)
            self.img_dir = img_dir
            self.transform = transform

        def __len__(self): return len(self.df)

        def __getitem__(self, idx):
            row  = self.df.iloc[idx]
            path = os.path.join(self.img_dir, row['image'] + '.jpg')
            img  = Image.open(path).convert('RGB')
            label = CLASS_ORDER.index(row['dx'])
            return self.transform(img), label

    isic_ds     = ISIC2019Dataset(isic_sample, ISIC_IMG_DIR, val_transform)
    isic_loader = DataLoader(isic_ds, batch_size=32, shuffle=False,
                             num_workers=2, pin_memory=True)

    model.eval()
    ood_preds, ood_labels = [], []
    with torch.no_grad():
        for imgs, labels in isic_loader:
            imgs = imgs.to(DEVICE)
            ood_preds.extend(model(imgs).argmax(1).cpu().numpy())
            ood_labels.extend(labels.numpy())

    ood_acc     = (np.array(ood_preds) == np.array(ood_labels)).mean()
    ood_bal_acc = balanced_accuracy_score(ood_labels, ood_preds)
    ood_f1      = f1_score(ood_labels, ood_preds, average='macro')

    print('=== ISIC 2019 OOD Results ===')
    print(f'  Images evaluated  : {len(ood_labels)}')
    print(f'  Overall accuracy  : {ood_acc:.4f}')
    print(f'  Balanced accuracy : {ood_bal_acc:.4f}')
    print(f'  Macro F1          : {ood_f1:.4f}')
    print()
    print(classification_report(ood_labels, ood_preds,
                                target_names=[c.upper() for c in CLASS_ORDER]))
else:
    print('ISIC 2019 dataset not downloaded.')
    print('Manually download from https://challenge.isic-archive.com/data/#2019')
    print('and place Training_Input/ and GroundTruth.csv in isic2019/')

---
## 16. Inference Function
A single function to classify any dermoscopic image from a file path or URL.

In [ ]:
import urllib.request

def predict_image(image_source, model=model, top_k=3, show=True):
    """
    Classify a single dermatoscopic image.

    Args:
        image_source: file path (str) or PIL Image
        model: trained EfficientNet-B3
        top_k: number of top predictions to return
        show: whether to display the image with prediction

    Returns:
        list of (class_name, probability) tuples, sorted by probability
    """
    model.eval()

    if isinstance(image_source, str):
        img = Image.open(image_source).convert('RGB')
    else:
        img = image_source.convert('RGB')

    inp = val_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(inp)
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    results = sorted(
        [(LABEL_MAP[CLASS_ORDER[i]], float(probs[i])) for i in range(len(CLASS_ORDER))],
        key=lambda x: x[1], reverse=True
    )

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(img)
        axes[0].set_title('Input Image', fontweight='bold')
        axes[0].axis('off')

        names  = [r[0] for r in results[:top_k]]
        scores = [r[1] for r in results[:top_k]]
        bars   = axes[1].barh(names[::-1], scores[::-1],
                               color=['#4e79a7']*top_k, edgecolor='white')
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Probability')
        axes[1].set_title(f'Top-{top_k} Predictions', fontweight='bold')
        for bar, score in zip(bars[::-1], scores[::-1]):
            axes[1].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                         f'{score:.1%}', va='center', fontsize=9)
        plt.tight_layout()
        plt.show()

    return results[:top_k]


# ── Example: run on a random test image ──────────────────────────────────────
sample_row  = test_df.sample(1, random_state=7).iloc[0]
sample_path = find_image(sample_row['image_id'])
true_class  = LABEL_MAP[sample_row['dx']]

print(f'True class: {true_class}')
top_preds = predict_image(sample_path, top_k=3)
print('\nTop predictions:')
for name, prob in top_preds:
    print(f'  {name:<30s} {prob:.1%}')

---
## 17. Results Summary


In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'{'Model':<25} {'Acc':>8} {'Bal Acc':>10} {'Macro F1':>10}')
print('-' * 60)
print(f'{'RF Baseline (Val)':<25} {rf_acc:>8.4f} {rf_bal_acc:>10.4f} {rf_macro_f1:>10.4f}')
print(f'{'EfficientNet-B3 (Val)':<25} {cnn_acc:>8.4f} {cnn_bal_acc:>10.4f} {cnn_macro_f1:>10.4f}')
print(f'{'EfficientNet-B3 (Test)':<25} {test_acc:>8.4f} {test_bal_acc:>10.4f} {test_macro_f1:>10.4f}')
print(f'{'  + TTA (Test)':<25} {tta_acc:>8.4f} {tta_bal_acc:>10.4f} {tta_f1:>10.4f}')
try:
    print(f'{'ISIC 2019 OOD':<25} {ood_acc:>8.4f} {ood_bal_acc:>10.4f} {ood_f1:>10.4f}')
except NameError:
    print(f'{'ISIC 2019 OOD':<25} {'N/A':>8} {'N/A':>10} {'N/A':>10}')
print('=' * 60)